# CSE 151B Competition — Starter Notebook

Welcome to the **CSE 151B Spring 2026 Math Reasoning Competition**!  
This notebook walks you through the full pipeline end-to-end:

1. Setting up the Python environment with `uv`
2. Loading the competition dataset
3. Running inference with **Qwen3-4B-Thinking** via vLLM (INT8 quantized)
4. Scoring responses against ground-truth answers
5. Saving results to JSONL for submission

The public dataset (`public.jsonl`) contains questions **with** answers so you can measure accuracy locally.  
The private test set used for the leaderboard does **not** include answers — for that, skip evaluation and submit the raw responses.

## 1. Environment Setup

We use [`uv`](https://github.com/astral-sh/uv) for fast, reproducible package management.

The steps below:
1. Install `uv` into `~/.local/bin`
2. Create a virtual environment at `.venv/`
3. Install all required packages (This might take a while)

> **After running this cell, restart the kernel** so that the newly installed packages (especially `vllm` and `transformers`) are picked up by the current Python session.

### Comment Out the cell below after first installation.

In [ ]:
# # Install uv
# !wget -qO- https://astral.sh/uv/install.sh | sh

# # Create a virtual environment
# !uv venv .venv --seed

# # Install dependencies — this is fast thanks to uv's parallel resolver
# !.venv/bin/python -m pip install sympy numpy transformers vllm tqdm bitsandbytes antlr4-python3-runtime==4.11.1 ipykernel jupyter

# # Install Jupyter Kernel
# !.venv/bin/python -m ipykernel install --user --name cse151b --display-name "Python (cse151b)"

# print("Done. Restart the kernel before proceeding.")
# print("Selection process: on top right, click on current kernel '(ususally named python)' -> 'select another kernel' -> 'Jupyter Kernel' -> 'Python (cse151b)'.")

In [ ]:
#!wget -qO- https://astral.sh/uv/install.sh | sh

downloading uv 0.11.13 x86_64-unknown-linux-gnu
installing to /usr/local/bin
  uv
  uvx
everything's installed!


In [ ]:
#!uv pip install torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 --index-url https://download.pytorch.org/whl/cu121

Using Python 3.12.13 environment at: /usr
Resolved 27 packages in 454ms
Uninstalled 16 packages in 210ms
Installed 16 packages in 192ms
 - nvidia-cublas-cu12==12.8.4.1
 + nvidia-cublas-cu12==12.1.3.1
 - nvidia-cuda-cupti-cu12==12.8.90
 + nvidia-cuda-cupti-cu12==12.1.105
 - nvidia-cuda-nvrtc-cu12==12.8.93
 + nvidia-cuda-nvrtc-cu12==12.1.105
 - nvidia-cuda-runtime-cu12==12.8.90
 + nvidia-cuda-runtime-cu12==12.1.105
 - nvidia-cudnn-cu12==9.10.2.21
 + nvidia-cudnn-cu12==9.1.0.70
 - nvidia-cufft-cu12==11.3.3.83
 + nvidia-cufft-cu12==11.0.2.54
 - nvidia-curand-cu12==10.3.9.90
 + nvidia-curand-cu12==10.3.2.106
 - nvidia-cusolver-cu12==11.7.3.90
 + nvidia-cusolver-cu12==11.4.5.107
 - nvidia-cusparse-cu12==12.5.8.93
 + nvidia-cusparse-cu12==12.1.0.106
 - nvidia-nccl-cu12==2.27.5
 + nvidia-nccl-cu12==2.21.5
 - nvidia-nvtx-cu12==12.8.90
 + nvidia-nvtx-cu12==12.1.105
 - sympy==1.14.0
 + sympy==1.13.1
 - torch==2.10.0
 + torch==2.5.1+cu121
 - torchaudio==2.10.0
 + torchaudio==2.5.1+cu121
 - torchvi

In [ ]:
#!uv pip install transformers==4.57.6 vllm==0.19.1 sympy numpy tqdm bitsandbytes antlr4-python3-runtime==4.11.1 ipykernel jupyter accelerate

Using Python 3.12.13 environment at: /usr
Resolved 245 packages in 70ms
Uninstalled 16 packages in 186ms
Installed 16 packages in 211ms
 - nvidia-cublas-cu12==12.1.3.1
 + nvidia-cublas-cu12==12.8.4.1
 - nvidia-cuda-cupti-cu12==12.1.105
 + nvidia-cuda-cupti-cu12==12.8.90
 - nvidia-cuda-nvrtc-cu12==12.1.105
 + nvidia-cuda-nvrtc-cu12==12.8.93
 - nvidia-cuda-runtime-cu12==12.1.105
 + nvidia-cuda-runtime-cu12==12.8.90
 - nvidia-cudnn-cu12==9.1.0.70
 + nvidia-cudnn-cu12==9.10.2.21
 - nvidia-cufft-cu12==11.0.2.54
 + nvidia-cufft-cu12==11.3.3.83
 - nvidia-curand-cu12==10.3.2.106
 + nvidia-curand-cu12==10.3.9.90
 - nvidia-cusolver-cu12==11.4.5.107
 + nvidia-cusolver-cu12==11.7.3.90
 - nvidia-cusparse-cu12==12.1.0.106
 + nvidia-cusparse-cu12==12.5.8.93
 - nvidia-nccl-cu12==2.21.5
 + nvidia-nccl-cu12==2.27.5
 - nvidia-nvtx-cu12==12.1.105
 + nvidia-nvtx-cu12==12.8.90
 - sympy==1.13.1
 + sympy==1.14.0
 - torch==2.5.1+cu121
 + torch==2.10.0
 - torchaudio==2.5.1+cu121
 + torchaudio==2.10.0
 - torchvi

### Run the cell below every time to activate the installed environment.

In [1]:
!pip uninstall -y vllm torch torchvision torchaudio xformers
!pip install -U uv
!uv pip install --system --no-cache vllm==0.19.1
!uv pip install --system --no-cache transformers==4.57.6 tqdm sympy numpy bitsandbytes antlr4-python3-runtime==4.11.1 accelerate

Found existing installation: torch 2.10.0+cu128
Uninstalling torch-2.10.0+cu128:
  Successfully uninstalled torch-2.10.0+cu128
Found existing installation: torchvision 0.25.0+cu128
Uninstalling torchvision-0.25.0+cu128:
  Successfully uninstalled torchvision-0.25.0+cu128
Found existing installation: torchaudio 2.10.0+cu128
Uninstalling torchaudio-2.10.0+cu128:
  Successfully uninstalled torchaudio-2.10.0+cu128
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.7/24.7 MB 80.6 MB/s eta 0:00:00
Using Python 3.12.13 environment at: /usr
Resolved 174 packages in 1.05s
Prepared 55 packages in 19.27s
Uninstalled 5 packages in 374ms
Installed 55 packages in 261ms
 + anthropic==0.102.0
 + apache-tvm-ffi==0.1.11
 + astor==0.8.1
 + blake3==1.0.8
 + cbor2==6.1.1
 + compressed-tensors==0.15.0.1
 + depyf==0.20.0
 + diskcache==5.6.3
 + dnspython==2.8.0
 + email-validator==2.3.0
 + fastapi-cli==0.0.24
 + fastapi-cloud-cli==0.17.1
 + fastar==0.11.0
 + flashinfer-cubin==0.6.6
 + flashinfer-python==0.6.6
 + 

## 2. Imports & Configuration

All key settings are collected in one place.  
- `DATA_PATH` — public dataset with ground-truth answers (use this to measure accuracy)
- `OUTPUT_PATH` — where per-question results will be written
- `GPU_ID` — which GPU to use (update if your machine has a different device index)
- `MAX_TOKENS` — maximum tokens the model may generate per response

In [4]:
import json
import os

# ── Configuration ─────────────────────────────────────────────────────────────
MODEL_ID    = "Qwen/Qwen3-4B-Thinking-2507"
GPU_ID      = "0"                    # CUDA_VISIBLE_DEVICES
DATA_PATH   = "/content/151B_SP26_Competition/data/public.jsonl"

OUTPUT_PATH = "results/starter_results.jsonl"
MAX_TOKENS  = 32768

os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID

import re
import sys
from pathlib import Path
from typing import Optional

from transformers import AutoTokenizer
from vllm import LLM, SamplingParams
from tqdm import tqdm

## 3. Load the Dataset

The dataset is stored as newline-delimited JSON (`.jsonl`). Each line is one question with the following fields:

| Field | Description |
|---|---|
| `id` | Unique question identifier |
| `question` | Problem statement |
| `options` | List of answer choices — present for **MCQ**, absent for **free-form** |
| `answer` | Ground-truth answer (letter for MCQ, value/list for free-form) |

In [5]:
data = [json.loads(line) for line in open(DATA_PATH)]

n_mcq  = sum(bool(d.get("options")) for d in data)
n_free = sum(not d.get("options")   for d in data)
print(f"Loaded {len(data)} questions  ({n_mcq} MCQ, {n_free} free-form)")

# Preview one MCQ and one free-form item
mcq_sample  = next(d for d in data if d.get("options"))
free_sample = next(d for d in data if not d.get("options"))

print("\n── MCQ sample ──")
print(json.dumps(mcq_sample, indent=2))
print("\n── Free-form sample ──")
print(json.dumps(free_sample, indent=2))

Loaded 1126 questions  (375 MCQ, 751 free-form)

── MCQ sample ──
{
  "question": "$int_{-infty}^{+infty} frac{a^{3/2}}{s^2+a^2} ds = $",
  "options": [
    "$0$",
    "$frac{1}{a}$",
    "$frac{3}{a}$",
    "$frac{1}{2a^2}$",
    "$frac{1}{2a}$",
    "$frac{2}{a}$",
    "$2a$",
    "$frac{3}{2a}$",
    "$frac{3}{2a^2}$",
    "$frac{1}{a^2}$"
  ],
  "answer": "F",
  "id": 1
}

── Free-form sample ──
{
  "question": "Find the sum of the first $325$ positive even whole numbers. Sum: [ANS]",
  "answer": [
    "325*(1+325)"
  ],
  "id": 0
}


## 4. Prompt Construction

We use two system prompts depending on the question type:

- **MCQ** — the model must select the best answer letter and wrap it in `\boxed{}`
- **Free-form** — the model solves step-by-step and puts the final answer in `\boxed{}`

`build_prompt()` returns the appropriate `(system, user)` pair for each item.

In [26]:
SYSTEM_PROMPT_MATH_BASELINE = (
    "You are an expert mathematician. Solve the problem step-by-step. "
    "Put your final answer inside \\boxed{}. "
    "If the problem has multiple sub-answers, separate them by commas inside a single \\boxed{}, "
    "e.g. \\boxed{3, 7}."
)

SYSTEM_PROMPT_MATH = (
    "You are an expert mathematician. "
    "Please reason step by step and solve the problem carefully. "
    "Put your final answer within \\boxed{}. "
    "If the problem has multiple [ANS] placeholders, give the answers in the same order, "
    "separated by commas inside a single \\boxed{}, e.g. \\boxed{3, 7}. "
    "Do not write [ANS] in your final answer."
    "Do not round decimal answers unless the problem explicitly asks for rounding."
)
SYSTEM_PROMPT_MCQ = (
    "You are an expert mathematician solving a multiple-choice math problem. "
    "Think carefully to determine the correct option. "
    "Do not put intermediate results in \\boxed{}. "
    "At the very end, output only one boxed capital letter. "
    "Do not include any explanation after the final answer. "
    "Example final format: \\boxed{C}."
)

SYSTEM_PROMPT_MCQ_BASELINE = (
    "You are an expert mathematician. "
    "Read the problem and the answer choices below, then select the single best answer. "
    "Output ONLY the letter of your chosen option inside \\boxed{}, e.g. \\boxed{C}."
)


# def build_prompt(question: str, options: Optional[list]) -> tuple[str, str]:
#     """Return (system_prompt, user_prompt) for a question."""
#     if options:
#         labels    = [chr(65 + i) for i in range(len(options))]
#         opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
#         return SYSTEM_PROMPT_MCQ, f"{question}\n\nOptions:\n{opts_text}"
#     return SYSTEM_PROMPT_MATH, question
def build_prompt(question, options=None):
    if options is None:
        num_ans = question.count("[ANS]")
        system = SYSTEM_PROMPT_MATH
        user = (
            f"{question}\n\n"
            f"There are {num_ans} [ANS] placeholder(s). "
            f"Give exactly {num_ans} final answer(s), in order, inside one final \\boxed{{}}."
        )
    else:
        system = SYSTEM_PROMPT_MCQ
        choices = "\n".join(
            f"{chr(65+i)}. {opt}" for i, opt in enumerate(options)
        )
        user = (
            f"{question}\n\n"
            f"Answer choices:\n{choices}\n\n"
            "Choose exactly one option letter."
        )
    return system, user

# Verify with samples
for label, item in [("MCQ", mcq_sample), ("Free-form", free_sample)]:
    sys_p, usr_p = build_prompt(item["question"], item.get("options"))
    print(f"── {label} user prompt (first 200 chars) ──")
    print(usr_p[:200], "...\n")

── MCQ user prompt (first 200 chars) ──
$int_{-infty}^{+infty} frac{a^{3/2}}{s^2+a^2} ds = $

Answer choices:
A. $0$
B. $frac{1}{a}$
C. $frac{3}{a}$
D. $frac{1}{2a^2}$
E. $frac{1}{2a}$
F. $frac{2}{a}$
G. $2a$
H. $frac{3}{2a}$
I. $frac{3}{2a ...

── Free-form user prompt (first 200 chars) ──
Find the sum of the first $325$ positive even whole numbers. Sum: [ANS]

There are 1 [ANS] placeholder(s). Give exactly 1 final answer(s), in order, inside one final \boxed{}. ...



## 5. Load Model with vLLM (for general case, vLLM is faster)

We load **Qwen3-4B-Thinking-2507** with **INT8 quantization** via BitsAndBytes.  
Setting `load_format="bitsandbytes"` tells vLLM to apply on-the-fly INT8 weight quantization, roughly halving GPU memory usage compared to BF16.

Key parameters:
- `gpu_memory_utilization` — fraction of GPU VRAM reserved for the model and KV cache
- `max_model_len` — maximum sequence length (prompt + generation)
- `max_num_seqs` — maximum number of sequences processed in parallel

In [7]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

llm = LLM(
    model=MODEL_ID,
    quantization="bitsandbytes",
    load_format="bitsandbytes",
    enable_prefix_caching=False,
    gpu_memory_utilization=0.75,
    max_model_len=16384,
    trust_remote_code=True,
    max_num_seqs=256,
    max_num_batched_tokens=32768,
)

sampling_params = SamplingParams(
    max_tokens=MAX_TOKENS,
    temperature=0.6,
    top_p=0.95,
    top_k=20,
    min_p=0.0,
    presence_penalty=0.0,
    repetition_penalty=1.0,
)

print("Model loaded.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

INFO 05-15 02:45:29 [utils.py:233] non-default args: {'trust_remote_code': True, 'load_format': 'bitsandbytes', 'max_model_len': 16384, 'enable_prefix_caching': False, 'gpu_memory_utilization': 0.75, 'max_num_batched_tokens': 32768, 'max_num_seqs': 256, 'disable_log_stats': True, 'quantization': 'bitsandbytes', 'model': 'Qwen/Qwen3-4B-Thinking-2507'}


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

INFO 05-15 02:45:53 [model.py:549] Resolved architecture: Qwen3ForCausalLM
INFO 05-15 02:45:53 [model.py:1678] Using max model len 16384
INFO 05-15 02:45:53 [scheduler.py:238] Chunked prefill is enabled with max_num_batched_tokens=32768.
INFO 05-15 02:45:53 [vllm.py:790] Asynchronous scheduling is enabled.


generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

WARNING 05-15 02:45:55 [system_utils.py:152] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized
Model loaded.


## 5. Load Model with Transformers (alternative to vLLM for DataHub)

We load **Qwen3-4B-Thinking-2507** with **INT4 quantization** via BitsAndBytes.  

Key parameters:
- `load_in_4bit` — quantization strategy of INT4

In [ ]:
# import torch
# from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# MODEL_ID = "Qwen/Qwen3-4B-Thinking-2507"

# tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
# tokenizer.pad_token = tokenizer.eos_token

# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_compute_dtype=torch.bfloat16,
#     bnb_4bit_use_double_quant=True,
# )

# llm = AutoModelForCausalLM.from_pretrained(
#     MODEL_ID,
#     trust_remote_code=True,
#     quantization_config=bnb_config,
#     device_map="auto",
# )


## 6. Generate Responses

We format every question into a chat-template prompt, then call `llm.generate()` in one batched pass.  
vLLM handles batching and scheduling internally — no manual batching needed.

In [8]:
import random

SEED = 151
N_SAMPLE = 100

rng = random.Random(SEED)

# sample fixed indices
sample_indices = rng.sample(range(len(data)), k=min(N_SAMPLE, len(data)))

# fixed sampled data
sample_data = [data[i] for i in sample_indices]

print(f"Sampled {len(sample_data)} questions with seed={SEED}")
print("First 10 sampled ids:", [item["id"] for item in sample_data[:10]])

Sampled 100 questions with seed=151
First 10 sampled ids: [838, 419, 721, 578, 756, 237, 944, 1002, 151, 937]


### Generate with vLLM

In [27]:
# Build prompts for first 5 entries
prompts = []
for item in sample_data:
    system, user = build_prompt(item["question"], item.get("options"))
    prompt_text = tokenizer.apply_chat_template(
        [{"role": "system", "content": system},
         {"role": "user",   "content": user}],
        tokenize=False,
        add_generation_prompt=True,
    )
    prompts.append(prompt_text)

# Generate
print(f"Generating responses for {len(prompts)} questions...")
outputs = llm.generate(prompts, sampling_params=sampling_params)

responses = [out.outputs[0].text.strip() for out in outputs]

# Preview first 3
for i in range(min(3, len(responses))):
    print(f"\n── Response {i} (id={data[i].get('id')}) ──")
    print(responses[i][:400], "..." if len(responses[i]) > 400 else "")

Generating responses for 100 questions...


Rendering prompts:   0%|          | 0/100 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/100 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]


── Response 0 (id=0) ──
Okay, let's try to figure out this problem step by step. So, first, Sylvia has a compact car with an 8-gallon tank. The car gets 33 miles per gallon. We need to find out how many miles it can travel when the tank is only 3/4 full. 

First, let me recall that if the tank is full, it can hold 8 gallons, right? So when it's 3/4 full, the amount of fuel in the tank would be 3/4 of 8 gallons. Let me ca ...

── Response 1 (id=1) ──
Okay, let's see. So the problem is about Cody buying 19 roses, and a dozen (which is 12 roses) costs $11.99. We need to find a fair price for 19 roses. Hmm, fair price probably means the exact cost without any discounts or anything, just the total cost based on the price per dozen.

First, let me recall that a dozen is 12 roses. So the cost for 12 roses is $11.99. Now, Cody wants 19 roses. So we n ...

── Response 2 (id=2) ──
Okay, let's try to figure out this problem step by step. The question is to complete the equation: cos(163 - 327) =

### Generate with Transformers (for Datahub)

In [ ]:
# # Build prompts for first 5 entries
# prompts = []
# for item in data[:5]:
#     system, user = build_prompt(item["question"], item.get("options"))
#     prompt_text = tokenizer.apply_chat_template(
#         [{"role": "system", "content": system},
#          {"role": "user",   "content": user}],
#         tokenize=False,
#         add_generation_prompt=True,
#     )
#     prompts.append(prompt_text)

# # Tokenize (padded batch)
# print(f"Generating responses for {len(prompts)} questions...")
# inputs = tokenizer(
#     prompts,
#     return_tensors="pt",
#     padding=True,
#     truncation=True,
#     max_length=16384,
# ).to(llm.device)

# # Generate
# with torch.no_grad():
#     output_ids = llm.generate(
#         **inputs,
#         max_new_tokens=MAX_TOKENS,
#         temperature=0.6,
#         top_p=0.95,
#         top_k=20,
#         repetition_penalty=1.0,
#         do_sample=True,
#     )

# # Decode only the new tokens (strip the prompt)
# responses = []
# for i, out in enumerate(output_ids):
#     new_tokens = out[inputs["input_ids"].shape[1]:]
#     responses.append(tokenizer.decode(new_tokens, skip_special_tokens=True).strip())

# # Preview first 3
# for i in range(min(3, len(responses))):
#     print(f"\n── Response {i} (id={data[i].get('id')}) ──")
#     print(responses[i][:400], "..." if len(responses[i]) > 400 else "")

## 7. Score Responses

Scoring differs by question type:

- **MCQ**: extract the predicted letter from `\boxed{}` and compare to the gold letter (exact match).
- **Free-form**: use `Judger.auto_judge()` which handles symbolic and numeric equivalence.

Each result record contains `{id, is_mcq, gold, response, correct}`.

In [10]:
%cd 151B_SP26_Competition/

/content/151B_SP26_Competition


In [11]:
!pwd

/content/151B_SP26_Competition


In [28]:
def extract_letter(text: str) -> str:
    m = re.search(r"\\boxed\{([A-Za-z])\}", text)
    if m:
        return m.group(1).upper()
    matches = re.findall(r"\b([A-Z])\b", text.upper())
    return matches[-1] if matches else ""


def score_mcq(response: str, gold_letter: str) -> bool:
    return extract_letter(response) == gold_letter.strip().upper()


# Load Judger for free-form scoring
sys.path.insert(0, ".")
from judger import Judger
judger = Judger(strict_extract=False)

results = []
for item, response in tqdm(zip(sample_data, responses), total=len(sample_data), desc="Scoring"):
    is_mcq = bool(item.get("options"))
    gold   = item["answer"]

    if is_mcq:
        correct = score_mcq(response, str(gold))
    else:
        gold_list = gold if isinstance(gold, list) else [gold]
        try:
            correct = judger.auto_judge(
                pred=response,
                gold=gold_list,
                options=[[]] * len(gold_list),
            )
        except Exception:
            correct = False

    results.append({
        "id":       item.get("id"),
        "is_mcq":   is_mcq,
        "gold":     gold,
        "response": response,
        "correct":  correct,
    })

print(f"Scoring complete. {len(results)} results.")

Scoring: 100%|██████████| 100/100 [00:06<00:00, 16.24it/s]

Scoring complete. 100 results.


## 8. Summary

Print accuracy broken down by question type.

In [29]:
mcq_res  = [r for r in results if r["is_mcq"]]
free_res = [r for r in results if not r["is_mcq"]]

def acc(subset):
    return sum(r["correct"] for r in subset) / len(subset) * 100 if subset else 0.0

print("=" * 50)
print("EVALUATION RESULTS")
print("=" * 50)
print(f"  MCQ        : {sum(r['correct'] for r in mcq_res):4d} / {len(mcq_res):4d}  ({acc(mcq_res):.2f}%)")
print(f"  Free-form  : {sum(r['correct'] for r in free_res):4d} / {len(free_res):4d}  ({acc(free_res):.2f}%)")
print(f"  Overall    : {sum(r['correct'] for r in results):4d} / {len(results):4d}  ({acc(results):.2f}%)")
print("=" * 50)

EVALUATION RESULTS
  MCQ        :   32 /   42  (76.19%)
  Free-form  :   31 /   58  (53.45%)
  Overall    :   63 /  100  (63.00%)


## 9. Save Results

Results are written as newline-delimited JSON.

**With evaluation** (public set — you have ground-truth):  
Each line: `{id, is_mcq, gold, response, correct}`

**Without evaluation** (private test set — no ground-truth available):  
Each line: `{id, is_mcq, response}` — omit `gold` and `correct`.

Toggle `SAVE_EVAL` below accordingly.

In [30]:
SAVE_EVAL = True   # Set to False when running on the private test set

out_path = Path(OUTPUT_PATH)
out_path.parent.mkdir(parents=True, exist_ok=True)

with open(out_path, "w") as f:
    for r in results:
        if SAVE_EVAL:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "gold": r["gold"],
                      "response": r["response"], "correct": r["correct"]}
        else:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "response": r["response"]}
        f.write(json.dumps(record) + "\n")

print(f"Saved {len(results)} records to {out_path}")

Saved 100 records to results/starter_results.jsonl


In [ ]:
import csv
from pathlib import Path

OUTPUT_CSV = "results/submission.csv"

rows = []
for item, response in zip(data, responses):
    rows.append({
        "id": item["id"],
        "response": response,
    })

out_path = Path(OUTPUT_CSV)
out_path.parent.mkdir(parents=True, exist_ok=True)

with open(out_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["id", "response"])
    writer.writeheader()
    writer.writerows(rows)

print(f"Saved {len(rows)} rows to {out_path}")

Saved 943 rows to results/submission.csv


## Next Steps

This notebook gives you a working baseline. Here are directions to improve your score:

- **Prompt engineering** — try different system prompts or few-shot examples inside the user turn
- **Sampling parameters** — adjust `temperature`, `top_p`, or use majority voting across multiple samples
- **Fine-tuning** — the competition allows model fine-tuning; see the course resources for guidance

Good luck!